### environment

In [ ]:
%pip install -U spacy spacy-transformers
%pip install -U spacy
%pip install --upgrade pip
%pip install pip setuptools wheel
!python -m spacy download en_core_web_sm 

In [1]:
import os, rich, json
from tqdm import tqdm
import spacy
from spacy.tokens import DocBin
from label_models import Label

In [ ]:
# spacy.prefer_gpu()

False

In [2]:
nlp = spacy.load("en_core_web_sm") 
doc_bin = DocBin()

/home/ronji/repos/reverse-ats/ner-spacy-dev/ner-spacy/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### processing

In [ ]:
def split_doc(doc, labels, MAX_TOKENS=500, MIN_TOKENS = 10): 
    sequences = []
    current_start = 0 
    '''
    split a doc object into sequences short enough (MAX_TOKENS) to be used in spacy's model training
    MIN_TOKENS: to create reasonably long sequences in case of last few remaining tokens
    '''
    while current_start < len(doc):
        current_end = min(current_start + MAX_TOKENS, len(doc)) 
        subdoc = doc[current_start:current_end] # spacy Doc allows slicing, used not to run nlp() twice due to runtime
        subdoc_start_char = subdoc[0].idx 
        subdoc_end_char = subdoc[-1].idx + len(subdoc[-1])
        seq_labels = [
            Label(
                label['start'] - subdoc_start_char,
                label['end'] - subdoc_start_char,
                label['label'],
                label['value']
            )
            for label in labels # iterate labels only in the current chunk
            if label["start"] >= subdoc_start_char and label["end"] <= subdoc_end_char
        ]
        if len(subdoc) > MIN_TOKENS or len(seq_labels) > 0:
            sequences.append((subdoc, seq_labels))
        current_start = current_end
    return sequences


'''for token in doc:
        current_seq.append(token)
        if len(current_seq) >= MAX_TOKENS:
            seq_text = "".join(     # join all tokens in sequence into a doc again
                [t.text_with_ws for t in current_seq])      # whitespace included to preserve label offsets
            seq_labels = [ 
                Label(
                    label['start'] - current_offset, # adjust label indexes to preserve original data
                    label['end'] - current_offset,
                    label['label'],
                    label['value']
                )
                for label in labels # iterate labels only in the current chunk
                if label["start"] >= current_offset and label["end"] <= current_offset + len(seq_text)
            ]
            sequences.append((seq_text, seq_labels)) 
            # for debugging purposes; source_id is same for all sequences of origin doc
            current_offset += len(seq_text)
            current_seq = [] # reset for next sequence' tokens
    
    if current_seq: # handle remaining tokens
        seq_text = "".join([t.text_with_ws for t in current_seq])
        seq_labels = [
            Label(
                label['start'] - current_offset, 
                label['end'] - current_offset,
                label['label'],
                label['value']
            )
            for label in labels 
            if label["start"] >= current_offset and label["end"] <= current_offset + len(seq_text)
        ]
        if len(current_seq) > MIN_TOKENS or len(seq_labels) > 0:
            sequences.append((seq_text, seq_labels))'''


'\n    split a document with its labels into sequences\n    MAX_TOKENS: due to spacy sequence length being limited by 512 tokens, and alignment mode adding a bit in some cases, set to 500\n    MIN_TOKENS: to create reasonably long sequence in case there are remaining tokens after the loop\n\n    for token in doc:\n        current_seq.append(token)\n        if len(current_seq) >= MAX_TOKENS:\n            seq_text = "".join(     # join all tokens in sequence into a doc again\n                [t.text_with_ws for t in current_seq])      # whitespace included to preserve label offsets\n            seq_labels = [ \n                Label(\n                    label[\'start\'] - current_offset, # adjust label indexes to preserve original data\n                    label[\'end\'] - current_offset,\n                    label[\'label\'],\n                    label[\'value\']\n                )\n                for label in labels # iterate labels only in the current chunk\n                if label

In [ ]:
with open(f'{os.getcwd()}/labels_en.json','r') as f:
    dataset = json.loads(f.read())

doc_bin = DocBin()

empty_span_count, overlap_count, ent_count = 0,0,0 

for doc in tqdm(dataset):
    source_id, content, labels = doc['id'], doc['content'], doc['labels']
    doc = nlp(content)
    sequences = split_doc(doc,labels)

    for seq_text, seq_labels in sequences:
        doc = nlp(seq_text)
        ents = []
        for l in seq_labels:
            start, end, label, value = l.start, l.end, l.label, l.value
            span = doc.char_span(start,end,label,alignment_mode='expand')
            if span is None:
                empty_span_count += 1
                continue
            if any(
                    span.start < existing_span.end and 
                    span.end > existing_span.start 
                for existing_span in ents):
                overlap_count += 1
                continue
            ents.append(span)
            ent_count += len(ents)
        doc.ents = ents
        doc_bin.add(doc)

doc_bin.to_disk("train.spacy")
print(f'final entities: {ent_count}\noverlaps: {overlap_count}\nempty spans:\n{empty_span_count}')
# after trying multiple ways of processing the data, this proved to be the most effective for entity count


 60%|██████    | 3/5 [00:00<00:00, 13.02it/s]

1143
469
964


100%|██████████| 5/5 [00:00<00:00, 11.68it/s]

1157
947
